<a href="https://colab.research.google.com/github/smstrong920/GB885-Final-Project---Strong---S/blob/main/GB885_Final_Project_Ingestion_and_Cleaning_Strong_S.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#GB885 Final Porject - Rush Sportswear - Ingesting and cleaning file
####I will be bringing in the data sources and cleaning them in this workbook. Analysis will take place in a seperate workbook

In [1]:
#Import needed modules
import pandas as pd
from google.colab import files

In [2]:
#load sales, retailer, and product tables from local files
uploaded = files.upload()

In [3]:
#Create pandas dataframes for each of the files
#this first file is pipe delimited so I need to specify the seperator
products = pd.read_csv('/content/TABLE_PRODUCTS_885.csv', sep='|')
#The last two are comma seperated so they uploaded fine with the basic command
retailers = pd.read_csv('/content/TABLE_RETAILER_885.csv')
sales = pd.read_csv('/content/TABLE_SALES_885.csv')

In [4]:
#preview one by one to ensure they were uploaded
products.head(10)

,PRODUCT_ID,PRODUCT_NAME
0,20,Men's Street Footwear
1,30,Men's Athletic Footwear
2,120,Women's Street Footwear
3,130,Women's Athletic Footwear
4,40,Men's Apparel
5,140,Women's Apparel


In [5]:
retailers.head(10)

,RETAILER_ID,RETAILER,REGION,STATE,CITY
0,A00MOHCO,Amazon,Midwest,Ohio,Columbus
1,A00NMAPO,Amazon,Northeast,Maine,Portland
2,A00NMABO,Amazon,Northeast,Massachusetts,Boston
3,A00NNEMA,Amazon,Northeast,New Hampshire,Manchester
4,A00NVEBU,Amazon,Northeast,Vermont,Burlington
5,A00SALBI,Amazon,South,Alabama,Birmingham
6,A00SKELO,Amazon,Southeast,Kentucky,Louisville
7,A00SNOCH,Amazon,Southeast,North Carolina,Charlotte
8,A00WALAN,Amazon,West,Alaska,Anchorage
9,F00MILCH,Foot Locker,Midwest,Illinois,Chicago


In [6]:
sales.head(10)

,ORDER_ID,RETAILER_ID,INVOICE_DATE,MONTH,DAY,YEAR,PRODUCT_ID,PRICE_PER_UNIT,UNITS_SOLD,OPERATING_MARGIN,SALES_METHOD
0,1,A00MOHCO,1/1/2020,1,1,2020,20,50.0,1200,0.5,In-store
1,7,A00MOHCO,1/7/2020,1,7,2020,20,50.0,1250,0.5,In-store
2,13,A00MOHCO,1/25/2020,1,25,2020,20,50.0,1220,0.5,Outlet
3,19,A00MOHCO,1/31/2020,1,31,2020,20,50.0,1200,0.5,Outlet
4,25,A00MOHCO,2/6/2020,2,6,2020,20,60.0,1220,0.5,Outlet
5,31,A00MOHCO,3/4/2020,3,4,2020,20,60.0,1250,0.5,Outlet
6,37,A00MOHCO,3/10/2020,3,10,2020,20,60.0,1275,0.5,Outlet
7,43,A00MOHCO,3/16/2020,3,16,2020,20,60.0,1250,0.5,Outlet
8,49,A00MOHCO,4/19/2020,4,19,2020,20,60.0,1200,0.5,Outlet
9,57,A00MOHCO,4/27/2020,4,27,2020,20,65.0,1150,0.5,Outlet


####There seem to be keys that I can use to cleanly join these into one dataset. I will just have to make sure data types match between dataframes and check row counts before and after to make sure I'm not duplicating or dropping anything on the joins

In [7]:
#Explore data type of keys and see if there are any nulls
products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   PRODUCT_ID    6 non-null      int64 
 1   PRODUCT_NAME  6 non-null      object
dtypes: int64(1), object(1)
memory usage: 228.0+ bytes


In [8]:
sales.info()
#invoice date shoudl be a date not an object, units sold should be a intedger not an object

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9648 entries, 0 to 9647
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ORDER_ID          9648 non-null   int64  
 1   RETAILER_ID       9648 non-null   object 
 2   INVOICE_DATE      9648 non-null   object 
 3   MONTH             9648 non-null   int64  
 4   DAY               9648 non-null   int64  
 5   YEAR              9648 non-null   int64  
 6   PRODUCT_ID        9648 non-null   int64  
 7   PRICE_PER_UNIT    9646 non-null   float64
 8   UNITS_SOLD        9648 non-null   object 
 9   OPERATING_MARGIN  9648 non-null   float64
 10  SALES_METHOD      9648 non-null   object 
dtypes: float64(2), int64(5), object(4)
memory usage: 829.3+ KB


In [9]:
retailers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 110 entries, 0 to 109
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   RETAILER_ID  110 non-null    object
 1   RETAILER     110 non-null    object
 2   REGION       110 non-null    object
 3   STATE        110 non-null    object
 4   CITY         110 non-null    object
dtypes: object(5)
memory usage: 4.4+ KB


#### I will convert all the ID keys to objects to get a clean join. There are no null values in the fields but I will also check for other common placeholders for null values within the keys. As well as potential duplicates

In [10]:
# It didn't appear from the info function that we had nulls in the keys. But running an additional check
print("sales key nulls:")
print(sales[['PRODUCT_ID', 'RETAILER_ID']].isna().sum())
print()

print("products key nulls:")
print(products[['PRODUCT_ID']].isna().sum())
print()

print("retailers key nulls:")
print(retailers[['RETAILER_ID']].isna().sum())
#confirmed no explicit nulls

sales key nulls:
PRODUCT_ID     0
RETAILER_ID    0
dtype: int64

products key nulls:
PRODUCT_ID    0
dtype: int64

retailers key nulls:
RETAILER_ID    0
dtype: int64


In [11]:
# printing distinct values in each key field to determine if there are any unconventional nulls
print("SALES - unique PRODUCT_ID:")
print(sorted(sales['PRODUCT_ID'].unique()))
print()

print("SALES - unique RETAILER_ID:")
print(sorted(sales['RETAILER_ID'].unique()))
print()

print("PRODUCTS - unique PRODUCT_ID:")
print(sorted(products['PRODUCT_ID'].unique()))
print()

print("RETAILERS - unique RETAILER_ID:")
print(sorted(retailers['RETAILER_ID'].unique()))
#There is one apparent null placeholder in sales in sales '999999999'
#also noting that product id fields are still integers so those should be switched to objects

SALES - unique PRODUCT_ID:
[np.int64(20), np.int64(30), np.int64(40), np.int64(120), np.int64(130), np.int64(140)]

SALES - unique RETAILER_ID:
['999999999', 'A00MOHCO', 'A00NMABO', 'A00NMAPO', 'A00NNEMA', 'A00NVEBU', 'A00SALBI', 'A00SKELO', 'A00SNOCH', 'A00WALAN', 'F00MILCH', 'F00MIODE', 'F00MKAWI', 'F00MMIDE', 'F00MMIMI', 'F00MMIST', 'F00MNEOM', 'F00MNOFA', 'F00MSOSI', 'F00NCOHA', 'F00NDEWI', 'F00NMABA', 'F00NNEMA', 'F00NNENE', 'F00NPEPH', 'F00NRHPR', 'F00NWECH', 'F00SFLMI', 'F00SGEAT', 'F00SKELO', 'F00SLONE', 'F00SMIJA', 'F00SSOCH', 'F00STEDA', 'F00STEKN', 'F00SVIRI', 'F00WALAN', 'F00WARPH', 'F00WCALO', 'F00WHAHO', 'F00WIDBO', 'F00WWASE', 'F00WWYCH', 'K00MKAWI', 'K00MMIMI', 'K00MMOBI', 'K00NDEWI', 'K00NNEAL', 'K00NNENE', 'K00SOKOK', 'K00WARPH', 'K00WCALO', 'K00WCASA', 'K00WNEAL', 'K00WWYCH', 'S00MILCH', 'S00MMIDE', 'S00MMOBI', 'S00MNEOM', 'S00MNOFA', 'S00MSOSI', 'S00NCOHA', 'S00NMABA', 'S00NMABO', 'S00NNENE', 'S00NRHPR', 'S00SALBI', 'S00SFLMI', 'S00SFLOR', 'S00SGEAT', 'S00SMIJA', 'S

In [12]:
#view row with null placeholder
sales[sales['RETAILER_ID'] == '999999999']
#this seems like a legitimate sale, no other data in the row is missing
#I will not be able to use this to answer retailer or state questions

,ORDER_ID,RETAILER_ID,INVOICE_DATE,MONTH,DAY,YEAR,PRODUCT_ID,PRICE_PER_UNIT,UNITS_SOLD,OPERATING_MARGIN,SALES_METHOD
1446,8668,999999999,7/23/2021,7,23,2021,20,60.0,298,0.42,Outlet


In [13]:
#checking for duplicate keys across tables
print("Duplicate PRODUCT_ID in products:", products['PRODUCT_ID'].duplicated().sum())
print("Duplicate RETAILER_ID in retailers:", retailers['RETAILER_ID'].duplicated().sum())
print("Duplicate ORDER_ID in sales:", sales['ORDER_ID'].duplicated().sum())
#we have 4 duplicate retailer_id values in retailers

Duplicate PRODUCT_ID in products: 0
Duplicate RETAILER_ID in retailers: 4
Duplicate ORDER_ID in sales: 0


In [14]:
#inspect duplicates
dupes = retailers[retailers['RETAILER_ID'].duplicated(keep=False)].sort_values('RETAILER_ID')
dupes
#these 4 keys aren't actually unique as they are each assigned to two different stores
#There are no additional fields in the sales table that would allow me to distinguish which one is which.
#Depending on the grain we are looking at I will have to drop certain rows. Example for ID S00NNENE I can't distinguish which rows belong to NJ and which to NY



,RETAILER_ID,RETAILER,REGION,STATE,CITY
63,S00NNENE,Sports Direct,Northeast,New Jersey,Newark
64,S00NNENE,Sports Direct,Northeast,New York,New York
81,W00SARLI,Walmart,South,Arkansas,Little Rock
97,W00SARLI,West Gear,South,Arkansas,Little Rock
84,W00SFLOR,Walmart,Southeast,Florida,Orlando
102,W00SFLOR,West Gear,Southeast,Florida,Orlando
83,W00STEHO,Walmart,South,Texas,Houston
100,W00STEHO,West Gear,South,Texas,Houston


####Because I won't be able to distinguish some of these stores from another because of shared keys I will have to create multiple dataframes to work with and strategically drop some of these stores that I can't attribute sales to correctly. Because of this my strategy is changing to do as much cleaning as I can pre merge so I don't have to duplicate work after I join datasets together.

In [15]:
#Searching for nulls across tables
sales.isna().sum()
#There are two null price per unit fields

,0
ORDER_ID,0
RETAILER_ID,0
INVOICE_DATE,0
MONTH,0
DAY,0
YEAR,0
PRODUCT_ID,0
PRICE_PER_UNIT,2
UNITS_SOLD,0
OPERATING_MARGIN,0


In [16]:
# Explore nulls
sales[sales['PRICE_PER_UNIT'].isna()]

,ORDER_ID,RETAILER_ID,INVOICE_DATE,MONTH,DAY,YEAR,PRODUCT_ID,PRICE_PER_UNIT,UNITS_SOLD,OPERATING_MARGIN,SALES_METHOD
98,591,A00NVEBU,4/2/2020,4,2,2020,20,NaN,525,0.35,In-store
99,597,A00NVEBU,4/8/2020,4,8,2020,20,NaN,525,0.50,In-store


In [17]:
#These are both product ID 20. Are all the values the same in this field?
sorted(sales[sales['PRODUCT_ID'] == 20]['PRICE_PER_UNIT'].dropna().unique())
#THey aren't and there appears to be another null fill in value of 99999 that we will have to deal with

[np.float64(7.0),
 np.float64(9.0),
 np.float64(10.0),
 np.float64(12.0),
 np.float64(13.0),
 np.float64(14.0),
 np.float64(15.0),
 np.float64(16.0),
 np.float64(17.0),
 np.float64(18.0),
 np.float64(19.0),
 np.float64(20.0),
 np.float64(21.0),
 np.float64(22.0),
 np.float64(23.0),
 np.float64(24.0),
 np.float64(25.0),
 np.float64(26.0),
 np.float64(27.0),
 np.float64(28.0),
 np.float64(29.0),
 np.float64(30.0),
 np.float64(31.0),
 np.float64(32.0),
 np.float64(33.0),
 np.float64(34.0),
 np.float64(35.0),
 np.float64(36.0),
 np.float64(37.0),
 np.float64(38.0),
 np.float64(39.0),
 np.float64(40.0),
 np.float64(41.0),
 np.float64(42.0),
 np.float64(43.0),
 np.float64(44.0),
 np.float64(45.0),
 np.float64(46.0),
 np.float64(47.0),
 np.float64(48.0),
 np.float64(49.0),
 np.float64(50.0),
 np.float64(51.0),
 np.float64(52.0),
 np.float64(53.0),
 np.float64(54.0),
 np.float64(55.0),
 np.float64(56.0),
 np.float64(57.0),
 np.float64(58.0),
 np.float64(59.0),
 np.float64(60.0),
 np.float64(61

In [18]:
#for the nulls and the placeholder null value of 99999 I'm going to impute values using the median
#First I need to exclude the null placeholder value from throwing the median off
valid_prices = sales[(sales['PRODUCT_ID'] == 20) & (sales['PRICE_PER_UNIT'] != 99999)]['PRICE_PER_UNIT']
median_price = valid_prices.median()
print(median_price)
#Median is 45 will impute using that

45.0


In [19]:
products.isna().sum()
#no explicit nulls in this table

,0
PRODUCT_ID,0
PRODUCT_NAME,0


In [20]:
retailers.isna().sum()
#no explicit nulls in this table


,0
RETAILER_ID,0
RETAILER,0
REGION,0
STATE,0
CITY,0


In [21]:
#Creating a list of numeric fields and checking for any non numeric values
numeric_cols = ['PRICE_PER_UNIT', 'UNITS_SOLD', 'OPERATING_MARGIN']

for col in numeric_cols:
    non_numeric = sales[pd.to_numeric(sales[col], errors='coerce').isna() & sales[col].notna()]
    print(f"{col}: {len(non_numeric)} non-numeric values")
    if len(non_numeric) > 0:
        print(non_numeric[[col]])
    print()
#There are two units sold values that are null and have '***' as placeholders



PRICE_PER_UNIT: 0 non-numeric values

UNITS_SOLD: 2 non-numeric values
     UNITS_SOLD
1012        ***
1439        ***

OPERATING_MARGIN: 0 non-numeric values



In [22]:
#Explore these nulls
sales[sales['UNITS_SOLD'] == '***']

,ORDER_ID,RETAILER_ID,INVOICE_DATE,MONTH,DAY,YEAR,PRODUCT_ID,PRICE_PER_UNIT,UNITS_SOLD,OPERATING_MARGIN,SALES_METHOD
1012,6064,S00SALBI,5/27/2021,5,27,2021,20,51.0,***,0.45,Online
1439,8626,W00MIODE,12/10/2021,12,10,2021,20,29.0,***,0.46,Outlet


In [23]:
#Did these retailers have similar sales on the same products?
sales[(sales['RETAILER_ID'] == 'S00SALBI') & (sales['PRODUCT_ID'] == 20)]

,ORDER_ID,RETAILER_ID,INVOICE_DATE,MONTH,DAY,YEAR,PRODUCT_ID,PRICE_PER_UNIT,UNITS_SOLD,OPERATING_MARGIN,SALES_METHOD
985,5902,S00SALBI,2/4/2021,2,4,2021,20,23.0,163,0.40,Online
986,5908,S00SALBI,3/6/2021,3,6,2021,20,28.0,156,0.45,Online
987,5914,S00SALBI,4/5/2021,4,5,2021,20,19.0,195,0.47,Online
988,5920,S00SALBI,5/5/2021,5,5,2021,20,18.0,203,0.50,Online
989,5926,S00SALBI,6/4/2021,6,4,2021,20,42.0,195,0.49,Online
990,5932,S00SALBI,7/6/2021,7,6,2021,20,48.0,217,0.50,Online
991,5938,S00SALBI,8/8/2021,8,8,2021,20,46.0,210,0.55,Online
992,5944,S00SALBI,9/5/2021,9,5,2021,20,50.0,176,0.50,Online
993,5950,S00SALBI,10/4/2021,10,4,2021,20,36.0,167,0.51,Online
994,5956,S00SALBI,11/5/2021,11,5,2021,20,33.0,182,0.55,Online


In [24]:
sales[(sales['RETAILER_ID'] == 'W00MIODE') & (sales['PRODUCT_ID'] == 20) & (sales['SALES_METHOD'] == 'Outlet')]

,ORDER_ID,RETAILER_ID,INVOICE_DATE,MONTH,DAY,YEAR,PRODUCT_ID,PRICE_PER_UNIT,UNITS_SOLD,OPERATING_MARGIN,SALES_METHOD
1432,8584,W00MIODE,5/12/2021,5,12,2021,20,27.0,228,0.43,Outlet
1433,8590,W00MIODE,6/11/2021,6,11,2021,20,21.0,208,0.43,Outlet
1434,8596,W00MIODE,7/10/2021,7,10,2021,20,28.0,248,0.43,Outlet
1435,8602,W00MIODE,8/11/2021,8,11,2021,20,27.0,248,0.47,Outlet
1436,8608,W00MIODE,9/12/2021,9,12,2021,20,23.0,228,0.45,Outlet
1437,8614,W00MIODE,10/11/2021,10,11,2021,20,24.0,201,0.47,Outlet
1438,8620,W00MIODE,11/11/2021,11,11,2021,20,22.0,165,0.43,Outlet
1439,8626,W00MIODE,12/10/2021,12,10,2021,20,29.0,***,0.46,Outlet


####Both of these retailers have consistently sold similar amounts of this product from month to month I will impute the median into these null fields

In [25]:
#Find median values
# Median for row 1012 context — S00SALBI, Product 20, Online
context_1012 = sales[(sales['RETAILER_ID'] == 'S00SALBI') &
                      (sales['PRODUCT_ID'] == 20) &
                      (sales['SALES_METHOD'] == 'Online') &
                      (sales['UNITS_SOLD'] != '***')]
median_1012 = pd.to_numeric(context_1012['UNITS_SOLD'], errors='coerce').median()
print("Median for row 1012 context:", median_1012)

# Median for row 1439 context — W00MIODE, Product 20, Outlet
context_1439 = sales[(sales['RETAILER_ID'] == 'W00MIODE') &
                      (sales['PRODUCT_ID'] == 20) &
                      (sales['SALES_METHOD'] == 'Outlet') &
                      (sales['UNITS_SOLD'] != '***')]
median_1439 = pd.to_numeric(context_1439['UNITS_SOLD'], errors='coerce').median()
print("Median for row 1439 context:", median_1439)

Median for row 1012 context: 165.0
Median for row 1439 context: 228.0


In [26]:
##Looking for more outliers like the 9999 values we have been seeing
print(pd.to_numeric(sales['UNITS_SOLD'], errors='coerce').describe())
print()
print(sales['OPERATING_MARGIN'].describe())
# Don't see any additional 9999 values but do see a 0 which I should investigate

count    9646.000000
mean      256.943811
std       214.271313
min         0.000000
25%       106.000000
50%       176.000000
75%       350.000000
max      1275.000000
Name: UNITS_SOLD, dtype: float64

count    9648.000000
mean        0.422991
std         0.097197
min         0.100000
25%         0.350000
50%         0.410000
75%         0.490000
max         0.800000
Name: OPERATING_MARGIN, dtype: float64


In [27]:
sales[pd.to_numeric(sales['UNITS_SOLD'], errors='coerce') == 0]
#All the same product around the same timeframe

,ORDER_ID,RETAILER_ID,INVOICE_DATE,MONTH,DAY,YEAR,PRODUCT_ID,PRICE_PER_UNIT,UNITS_SOLD,OPERATING_MARGIN,SALES_METHOD
6603,1020,F00SVIRI,6/5/2021,6,5,2021,130,35.0,0,0.40,Outlet
6604,1026,F00SVIRI,6/11/2021,6,11,2021,130,30.0,0,0.40,Outlet
7250,4908,S00MMIDE,6/5/2021,6,5,2021,130,33.0,0,0.55,Online
7251,4914,S00MMIDE,6/11/2021,6,11,2021,130,27.0,0,0.53,Online


In [28]:
#Has this happened with other products
#no it hasn't
sales[pd.to_numeric(sales['UNITS_SOLD'], errors='coerce') == 0]['PRODUCT_ID'].value_counts()

,count
PRODUCT_ID,
130,4


In [29]:
#look at sales of the same product in June to see if there is some kind of pattern
sales[(sales['PRODUCT_ID'] == 130) & (sales['RETAILER_ID'].isin(['F00SVIRI', 'S00MMIDE']))].sort_values(['RETAILER_ID', 'INVOICE_DATE'])[['ORDER_ID', 'RETAILER_ID', 'INVOICE_DATE', 'PRICE_PER_UNIT', 'UNITS_SOLD', 'SALES_METHOD']]
#For ID F00SVIRI this seems like a genuine data quality issue this ID is generally selling hundreds of units per record. I will impute the median on the 0s for this one.
#ForS00MIDE sales are much more modest often in the single digits, I don't have enough evidence in this case to do any imputation I think

,ORDER_ID,RETAILER_ID,INVOICE_DATE,PRICE_PER_UNIT,UNITS_SOLD,SALES_METHOD
6623,1140,F00SVIRI,10/3/2021,45.0,450,Outlet
6624,1146,F00SVIRI,10/9/2021,65.0,550,Outlet
6596,978,F00SVIRI,4/24/2021,45.0,475,Outlet
6597,984,F00SVIRI,4/30/2021,55.0,400,Outlet
6599,996,F00SVIRI,5/12/2021,55.0,475,Outlet
6600,1002,F00SVIRI,5/18/2021,60.0,550,Outlet
6601,1008,F00SVIRI,5/24/2021,30.0,75,Outlet
6602,1014,F00SVIRI,5/30/2021,30.0,50,Outlet
6598,990,F00SVIRI,5/6/2021,40.0,375,Outlet
6604,1026,F00SVIRI,6/11/2021,30.0,0,Outlet


In [30]:
# PRODUCT_ID is still int64 here; UNITS_SOLD is an object column, so coerce before comparing
units_numeric = pd.to_numeric(sales['UNITS_SOLD'], errors='coerce')

context_svri = sales[(sales['RETAILER_ID'] == 'F00SVIRI') &
                     (sales['PRODUCT_ID'] == 130) &
                     (sales['SALES_METHOD'] == 'Outlet') &
                     (units_numeric != 0) &
                     (units_numeric.notna())]

median_svri = pd.to_numeric(context_svri['UNITS_SOLD'], errors='coerce').median()
print("Median for F00SVIRI:", median_svri, "| rows used:", len(context_svri))

Median for F00SVIRI: 375.0 | rows used: 27


In [31]:
#Get distinct values for all categorical data across tables
categorical_cols = {
    'sales': ['SALES_METHOD'],
    'retailers': ['REGION', 'STATE', 'CITY', 'RETAILER'],
    'products': ['PRODUCT_NAME']
}

dfs = {'sales': sales, 'retailers': retailers, 'products': products}

for table_name, cols in categorical_cols.items():
    df = dfs[table_name]
    for col in cols:
        print(f"=== {table_name}.{col} ===")
        print(df[col].value_counts())
        print()
        #There is a typo in sales method where at times 'Outlet' is misspelled as 'Ootlet'

=== sales.SALES_METHOD ===
SALES_METHOD
Online      4889
Outlet      2999
In-store    1740
Ootlet        20
Name: count, dtype: int64

=== retailers.REGION ===
REGION
Northeast    28
Midwest      24
West         24
South        18
Southeast    16
Name: count, dtype: int64

=== retailers.STATE ===
STATE
Florida           6
New York          5
Texas             5
California        4
Idaho             3
Mississippi       3
Maryland          3
Hawaii            3
Alaska            2
Illinois          2
Massachusetts     2
New Hampshire     2
Vermont           2
Alabama           2
Kentucky          2
North Carolina    2
Ohio              2
Maine             2
South Dakota      2
North Dakota      2
Nebraska          2
Missouri          2
Minnesota         2
Michigan          2
Kansas            2
Iowa              2
Louisiana         2
West Virginia     2
Rhode Island      2
Tennessee         2
Georgia           2
Pennsylvania      2
Delaware          2
Connecticut       2
Arizona         

#Cleaning Plan
#####-Convert keys to objects. Currently product_id is an int64 in sales and products tables. order_id is also an int in the sales table
#####-convert invoice date to a date field, currently an object

In [32]:
#converting integer keys to strings
sales['PRODUCT_ID'] = sales['PRODUCT_ID'].astype(str)
products['PRODUCT_ID'] = products['PRODUCT_ID'].astype(str)
sales['ORDER_ID'] = sales['ORDER_ID'].astype(str)

In [33]:
#confirm no more integer keys
sales.info()
products.info()
retailers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9648 entries, 0 to 9647
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ORDER_ID          9648 non-null   object 
 1   RETAILER_ID       9648 non-null   object 
 2   INVOICE_DATE      9648 non-null   object 
 3   MONTH             9648 non-null   int64  
 4   DAY               9648 non-null   int64  
 5   YEAR              9648 non-null   int64  
 6   PRODUCT_ID        9648 non-null   object 
 7   PRICE_PER_UNIT    9646 non-null   float64
 8   UNITS_SOLD        9648 non-null   object 
 9   OPERATING_MARGIN  9648 non-null   float64
 10  SALES_METHOD      9648 non-null   object 
dtypes: float64(2), int64(3), object(6)
memory usage: 829.3+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   PRODUCT_ID    6 non-null     

In [34]:
#convert invoice date from an object to a date
sales['INVOICE_DATE'] = pd.to_datetime(sales['INVOICE_DATE'])

In [35]:
#verify
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9648 entries, 0 to 9647
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ORDER_ID          9648 non-null   object        
 1   RETAILER_ID       9648 non-null   object        
 2   INVOICE_DATE      9648 non-null   datetime64[ns]
 3   MONTH             9648 non-null   int64         
 4   DAY               9648 non-null   int64         
 5   YEAR              9648 non-null   int64         
 6   PRODUCT_ID        9648 non-null   object        
 7   PRICE_PER_UNIT    9646 non-null   float64       
 8   UNITS_SOLD        9648 non-null   object        
 9   OPERATING_MARGIN  9648 non-null   float64       
 10  SALES_METHOD      9648 non-null   object        
dtypes: datetime64[ns](1), float64(2), int64(3), object(5)
memory usage: 829.3+ KB


In [36]:
#Need to convert  units sold from an object to an integer but first need to take care of imputing the medians in the null cases I identified earlier
# Plug in medians for the '***' rows
sales.loc[sales['ORDER_ID'] == '6064', 'UNITS_SOLD'] = median_1012
sales.loc[sales['ORDER_ID'] == '8626', 'UNITS_SOLD'] = median_1439

# Plug in median for F00SVIRI's two 0 rows
sales.loc[sales['ORDER_ID'].isin(['1020', '1026']), 'UNITS_SOLD'] = median_svri

# Now convert to numeric/int cleanly, no nulls left to worry about
sales['UNITS_SOLD'] = pd.to_numeric(sales['UNITS_SOLD']).astype(int)

In [37]:
#verify
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9648 entries, 0 to 9647
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ORDER_ID          9648 non-null   object        
 1   RETAILER_ID       9648 non-null   object        
 2   INVOICE_DATE      9648 non-null   datetime64[ns]
 3   MONTH             9648 non-null   int64         
 4   DAY               9648 non-null   int64         
 5   YEAR              9648 non-null   int64         
 6   PRODUCT_ID        9648 non-null   object        
 7   PRICE_PER_UNIT    9646 non-null   float64       
 8   UNITS_SOLD        9648 non-null   int64         
 9   OPERATING_MARGIN  9648 non-null   float64       
 10  SALES_METHOD      9648 non-null   object        
dtypes: datetime64[ns](1), float64(2), int64(4), object(4)
memory usage: 829.3+ KB


In [38]:
#Impute price per unit nulls and nulls represented by 9999 with median, this variable was assigned earlier
sales['PRICE_PER_UNIT'] = sales['PRICE_PER_UNIT'].fillna(median_price)
sales['PRICE_PER_UNIT'] = sales['PRICE_PER_UNIT'].replace(99999, median_price)

In [39]:
#verify 9999 placeholder is gone
sales['PRICE_PER_UNIT'].describe()


,PRICE_PER_UNIT
count,9648.000000
mean,45.212998
std,14.703776
min,7.000000
25%,35.000000
50%,45.000000
75%,55.000000
max,110.000000


In [40]:
#verify nulls are filled
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9648 entries, 0 to 9647
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ORDER_ID          9648 non-null   object        
 1   RETAILER_ID       9648 non-null   object        
 2   INVOICE_DATE      9648 non-null   datetime64[ns]
 3   MONTH             9648 non-null   int64         
 4   DAY               9648 non-null   int64         
 5   YEAR              9648 non-null   int64         
 6   PRODUCT_ID        9648 non-null   object        
 7   PRICE_PER_UNIT    9648 non-null   float64       
 8   UNITS_SOLD        9648 non-null   int64         
 9   OPERATING_MARGIN  9648 non-null   float64       
 10  SALES_METHOD      9648 non-null   object        
dtypes: datetime64[ns](1), float64(2), int64(4), object(4)
memory usage: 829.3+ KB


In [41]:
#convert sales method fields labeld 'Ootlet' to 'Outlet'
sales['SALES_METHOD'] = sales['SALES_METHOD'].replace('Ootlet', 'Outlet')

In [42]:
#confirm
sales['SALES_METHOD'].value_counts()

,count
SALES_METHOD,
Online,4889
Outlet,3019
In-store,1740


#####I'm going to create the dataframes I need now. Since we establsiehd that some retailers will not be able to be properly joined to their sales. I will have to create the following dataframes which will be slightly different and used to answer different questions
#####1. Product level dataframe. The join between products and sales should be 1 to 1 so I should be safe to do that and reuse that dataframe for analysis and other joins
#####2. State/region level data frame we will first drop the IDs that we can't attribute to a state/region. Either because they show up twice and could belong to two different states/regions, or they are missing '99999'. There are still some duplicated IDs which would cause the join to fan out. Getting around this by creating a distinct combination list of ID, region, and state before join. We will retain all the sales for each region/state without fanning on the join.
#####3. Similar strategy used to create a dataframe at the retailer level to answer questions there. Dropping the ambiguous ids that we can't attribute and then collapsing down to distinct retailer IDs so we don't create any duplicated sales in the join

In [43]:
# Merge sales + products into base dataframe for product-level analysis
print("sales rows before merge:", len(sales))

df_product_and_sales = sales.merge(products, on='PRODUCT_ID', how='left')

print("df_product rows after merge:", len(df_product_and_sales))
print("Row count match:", len(df_product_and_sales) == len(sales))

df_product_and_sales.head()

sales rows before merge: 9648
df_product rows after merge: 9648
Row count match: True


,ORDER_ID,RETAILER_ID,INVOICE_DATE,MONTH,DAY,YEAR,PRODUCT_ID,PRICE_PER_UNIT,UNITS_SOLD,OPERATING_MARGIN,SALES_METHOD,PRODUCT_NAME
0,1,A00MOHCO,2020-01-01,1,1,2020,20,50.0,1200,0.5,In-store,Men's Street Footwear
1,7,A00MOHCO,2020-01-07,1,7,2020,20,50.0,1250,0.5,In-store,Men's Street Footwear
2,13,A00MOHCO,2020-01-25,1,25,2020,20,50.0,1220,0.5,Outlet,Men's Street Footwear
3,19,A00MOHCO,2020-01-31,1,31,2020,20,50.0,1200,0.5,Outlet,Men's Street Footwear
4,25,A00MOHCO,2020-02-06,2,6,2020,20,60.0,1220,0.5,Outlet,Men's Street Footwear


In [44]:
# State-level dataframe: drop IDs we can't tie to a single state
bad_state_ids = ['999999999', 'S00NNENE']

# Retailer NAME is the ambiguous field for W00xxxx IDs, and state analysis
# doesn't need it — dropping it collapses those pairs into one clean row
geo_lookup = retailers[['RETAILER_ID', 'REGION', 'STATE']].drop_duplicates()
geo_lookup = geo_lookup[~geo_lookup['RETAILER_ID'].isin(bad_state_ids)]
#confirm dupes are dropped
print("Duplicate IDs left in lookup:", geo_lookup['RETAILER_ID'].duplicated().sum())

sales_for_state = df_product_and_sales[
    ~df_product_and_sales['RETAILER_ID'].isin(bad_state_ids)
]
print("Rows before merge:", len(sales_for_state))

df_state = sales_for_state.merge(geo_lookup, on='RETAILER_ID', how='inner')

print("Rows after merge:", len(df_state))
print("Row count match:", len(df_state) == len(sales_for_state))
print("Null states:", df_state['STATE'].isna().sum())
print("Null regions:", df_state['REGION'].isna().sum())

Duplicate IDs left in lookup: 0
Rows before merge: 9591
Rows after merge: 9591
Row count match: True
Null states: 0
Null regions: 0


In [45]:
# Retailer-level dataframe: drop IDs we can't tie to a single retailer
bad_retailer_ids = ['999999999', 'W00SARLI', 'W00SFLOR', 'W00STEHO']

# Mirror of the state logic: STATE/CITY are the ambiguous fields for S00NNENE,
# and retailer analysis doesn't need them — dropping them collapses that pair
retailer_lookup = retailers[['RETAILER_ID', 'RETAILER']].drop_duplicates()
retailer_lookup = retailer_lookup[~retailer_lookup['RETAILER_ID'].isin(bad_retailer_ids)]
# confirm dupes are dropped
print("Duplicate IDs left in lookup:", retailer_lookup['RETAILER_ID'].duplicated().sum())

sales_for_retailer = df_product_and_sales[
    ~df_product_and_sales['RETAILER_ID'].isin(bad_retailer_ids)
]
print("Rows before merge:", len(sales_for_retailer))

df_retailer = sales_for_retailer.merge(retailer_lookup, on='RETAILER_ID', how='inner')

print("Rows after merge:", len(df_retailer))
print("Row count match:", len(df_retailer) == len(sales_for_retailer))
print("Null retailers:", df_retailer['RETAILER'].isna().sum())
print("Distinct retailers:", df_retailer['RETAILER'].nunique())

Duplicate IDs left in lookup: 0
Rows before merge: 9080
Rows after merge: 9080
Row count match: True
Null retailers: 0
Distinct retailers: 6


In [46]:
#Converting dataframes to parquet files and downloading for use in analysis notebook
df_product_and_sales.to_parquet('df_product_and_sales.parquet')
df_state.to_parquet('df_state.parquet')
df_retailer.to_parquet('df_retailer.parquet')

files.download('df_product_and_sales.parquet')
files.download('df_state.parquet')
files.download('df_retailer.parquet')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>